# Snapshot Ensemble

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
import warnings
import copy
warnings.filterwarnings('ignore')

# Path
PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))
from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data as load_v1_original , load_test_data
from models.vanilla_rnn import VanillaRNN
from sklearn.metrics import mean_absolute_error

 # Load data
TRAIN_PATH = PROJECT_ROOT / "datasets_challenge" / "train"
TEST_PATH = PROJECT_ROOT / "datasets_challenge" / "test"
DIST_CSV = PROJECT_ROOT / "datasets_challenge" / "sideSlip_distribution.csv"
MODEL_DIR = PROJECT_ROOT / "expert_models"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42) # Use your existing reproducibility util

## Model Definition

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = VanillaRNN(
    input_size=22,          # number of features
    hidden_size=48,
    num_layers=1,
    dropout=0.0
).to(device)

criterion = nn.L1Loss()     # MAE (matches few-shot objective)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

In [3]:
CONFIG = {
    'TRAIN_PATH': PROJECT_ROOT / "datasets_challenge" / "train",
    'VAL_RATIO': 0.2,
    'BATCH_SIZE': 64,
    'COLUMNS_TO_DROP': ['sideSlip', 'vyCG', 'longitude', 'latitude', 'time'],
    'SEQUENCE_LENGTH': 20,
    'THRESHOLD': True,
    'THRESHOLD_VALUE': 3.0,
    'THRESHOLD_COLUMN': 'vxCG',
    'TARGET': 'sideSlip',
    'SCALING': True,
    'SEED': 42,
    'DIST_CSV': PROJECT_ROOT / "datasets_challenge" / "sideSlip_distribution.csv",
}

## Data loading 


In [4]:
print("Loading V1: Original Split")
train_v1, val_v1, pipeline_v1 = load_v1_original(
    train_path=CONFIG['TRAIN_PATH'],
    train_ratio=0.8,
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold=CONFIG['THRESHOLD'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    batchsize=CONFIG['BATCH_SIZE'],
    seed=CONFIG['SEED']
)

Loading V1: Original Split
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_001.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_002.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_003.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_004.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_005.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_006.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_007.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_008.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_009.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_010.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_011.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_012.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_013.csv
c:\mait_g2_drift_challenge\datasets_challenge\trai

## Snapshot Ensemble Training

In [5]:
EPOCHS = 50
SNAPSHOT_EPOCHS = [35, 40, 45, 50]   # epochs to save checkpoints
SAVE_DIR = PROJECT_ROOT / "snapshot_models"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

def train_with_snapshots(model, train_loader, val_loader):
    model.train()
    snapshot_paths = []

    for epoch in range(1, EPOCHS + 1):
        epoch_loss = 0.0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            preds = model(x)
            loss = criterion(preds.squeeze(), y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        epoch_loss /= len(train_loader)

        print(f"Epoch [{epoch}/{EPOCHS}] - Train MAE: {epoch_loss:.5f}")

        # Save snapshot
        if epoch in SNAPSHOT_EPOCHS:
            ckpt_path = SAVE_DIR / f"snapshot_epoch_{epoch}.pt"
            torch.save(model.state_dict(), ckpt_path)
            snapshot_paths.append(ckpt_path)
            print(f" Saved snapshot at epoch {epoch}")

    return snapshot_paths


In [6]:
# Train once, save multiple snapshots
snapshot_paths = train_with_snapshots(
    model,
    train_v1,
    val_v1
)

Epoch [1/50] - Train MAE: 0.31855
Epoch [2/50] - Train MAE: 0.20195
Epoch [3/50] - Train MAE: 0.18544
Epoch [4/50] - Train MAE: 0.17622
Epoch [5/50] - Train MAE: 0.17043
Epoch [6/50] - Train MAE: 0.16669
Epoch [7/50] - Train MAE: 0.16347
Epoch [8/50] - Train MAE: 0.16068
Epoch [9/50] - Train MAE: 0.15822
Epoch [10/50] - Train MAE: 0.15633
Epoch [11/50] - Train MAE: 0.15482
Epoch [12/50] - Train MAE: 0.15372
Epoch [13/50] - Train MAE: 0.15237
Epoch [14/50] - Train MAE: 0.15124
Epoch [15/50] - Train MAE: 0.14993
Epoch [16/50] - Train MAE: 0.14900
Epoch [17/50] - Train MAE: 0.14825
Epoch [18/50] - Train MAE: 0.14692
Epoch [19/50] - Train MAE: 0.14642
Epoch [20/50] - Train MAE: 0.14591
Epoch [21/50] - Train MAE: 0.14490
Epoch [22/50] - Train MAE: 0.14407
Epoch [23/50] - Train MAE: 0.14347
Epoch [24/50] - Train MAE: 0.14295
Epoch [25/50] - Train MAE: 0.14168
Epoch [26/50] - Train MAE: 0.14137
Epoch [27/50] - Train MAE: 0.14074
Epoch [28/50] - Train MAE: 0.14029
Epoch [29/50] - Train MAE: 0.

In [7]:
def predict(model, loader):
    model.eval()
    preds = []

    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            y_hat = model(x)
            preds.append(y_hat.cpu())

    return torch.cat(preds).numpy().flatten()


In [8]:
def snapshot_ensemble_predict(snapshot_paths, loader):
    all_preds = []

    for path in snapshot_paths:
        snapshot_model = VanillaRNN(
            input_size=22,
            hidden_size=48,
            num_layers=1,
            dropout=0.0
        ).to(device)

        snapshot_model.load_state_dict(torch.load(path, map_location=device))
        preds = predict(snapshot_model, loader)
        all_preds.append(preds)

    all_preds = np.stack(all_preds, axis=0)   # [num_snapshots, N]
    ensemble_preds = np.mean(all_preds, axis=0)

    return ensemble_preds


In [10]:
def extract_targets(loader):
    targets = []
    for _, y in loader:
        targets.append(y)
    return torch.cat(targets).numpy().flatten()

train_targets_v1 = extract_targets(train_v1)
val_targets_v1 = extract_targets(val_v1)

In [12]:
# Ground truth
y_val = extract_targets(val_v1)

# Ensemble prediction
ensemble_preds = snapshot_ensemble_predict(snapshot_paths, val_v1)

# MAE
mae = mean_absolute_error(y_val, ensemble_preds)
print(f"\nSnapshot Ensemble MAE: {mae:.5f}")



Snapshot Ensemble MAE: 0.13197


In [18]:
# Load complete training data (unshuffled) for visualization
# This uses the same preprocessing pipeline to ensure consistency
train_dataloader_unshuffled = load_test_data(
    test_path=TRAIN_PATH,
    batchsize=512,
    pipeline=pipeline_v1,  # Use the same preprocessing as training
    load_anonymized_data=False  # We have ground truth for training data
)

c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_001.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_002.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_003.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_004.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_005.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_006.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_007.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_008.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_009.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_010.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_011.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_012.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_013.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_014.csv
c:\mai

## Ground Truth vs Prediction

In [19]:
import plotly.graph_objects as go

# Set model to evaluation mode
model.eval()

# Storage for predictions and ground truth
total_loss = 0  
preds = []
ground_truths = []

# Generate predictions for all training data
with torch.no_grad():
    for step, (x, y) in enumerate(train_dataloader_unshuffled):
        X, y = x.to(device), y.to(device).reshape([len(y), 1])

        pred = model(X)
        loss = criterion(pred, y)
        total_loss += loss.item()
        
        # Store predictions and ground truth
        preds.append(pred.reshape(-1).cpu().numpy())
        ground_truths.append(y.reshape(-1).cpu().numpy())

    print(f"Avg. Loss: {total_loss/step}")
    
    # Concatenate all batches into single arrays
    ground_truths = np.concatenate(ground_truths)
    preds = np.concatenate(preds)

# Create interactive plot
fig = go.Figure()
fig.add_trace(go.Scatter(y=ground_truths, mode='lines', name='truth', line=dict(color='blue')))
fig.add_trace(go.Scatter(y=preds.flatten(), mode='lines', name='pred', line=dict(color='red'), opacity=0.8))
fig.update_layout(
    title='Ground Truth vs Predictions',
    xaxis_title='Sample Index',
    yaxis_title='Sideslip Angle (degrees)',
    width=1200,
    height=500
)
fig.show()

Avg. Loss: 0.13066058074322978


## Kaggle score of 0.30183